<a href="https://colab.research.google.com/github/antoine-bourgois/MIA_2026_UE16_NLP/blob/main/code/2_Cooccurrence_Tokens_Vecteurs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TP 2 : Représentation Tokens - Matrice de cooccurrence

- Charger un Dataset
- Description statistique du dataset
- Matrice de cooccurrence
- Mots les plus similaires
- Réduction de dimension
- Visualisation
- Évaluion modèle de similarité

### Installation des librairies Python utiles pour le TP

In [1]:
!pip install tqdm jupyter ipywidgets datasets pyarrow huggingface_hub pandas matplotlib numpy scipy scikit-learn umap-learn plotly --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from tqdm.auto import tqdm
from collections import Counter

### Charger un jeu de données

In [3]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

builder = load_dataset_builder(dataset_path)

builder = load_dataset_builder(dataset_path, )

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name!r:<20}{split.num_examples:<6,} documents")

Le jeu de données 'AntoineBourgois/Gallica_Hors-d-Usage_cleaned' contient 5 sous-ensembles :
    'tous'              6,156  documents
    'auteur_genre'      200    documents
    'auteur'            200    documents
    'date_publication'  200    documents
    'genre_litteraire'  200    documents


In [4]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
sous_ensemble, label_column = "date_publication", "date_publication"
# sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

fichier_local = hf_hub_download(
    repo_id=dataset_path,
    filename=f"data/{sous_ensemble}-00000-of-00001.parquet",
    repo_type="dataset",
    force_download=True,
)

table = pq.read_table(fichier_local)
dataset = table.to_pylist()

print("Nombre de documents :", len(dataset))


def conversions_string_vers_liste(string):
    if isinstance(string, str):
        return string.split(" ")


for document in dataset:
    document["tokens"] = conversions_string_vers_liste(document["tokens"])
    document["lemmes"] = conversions_string_vers_liste(document["lemmes"])

data/date_publication-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  178MB            

data/date_publication-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Nombre de documents : 200


### Structure du dataset

In [5]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

Type dataset: <class 'list'>
Longueur dataset: 200
Type 1er document: <class 'dict'>
Clés d'un document: dict_keys(['doc_id', 'date_publication', 'titre', 'auteur', 'auteur_genre', 'auteur_nom', 'auteur_prenom', 'genre_litteraire', 'texte', 'tokens', 'lemmes'])


In [6]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

doc_id               <class 'str'>    HDU-00027
date_publication     <class 'int'>    1801
titre                <class 'str'>    Atala, ou les Amours de deux sauvages dans le désert
auteur               <class 'str'>    François-René de Chateaubriand
auteur_genre         <class 'str'>    masculin
auteur_nom           <class 'str'>    Chateaubriand
auteur_prenom        <class 'str'>    François-René de
genre_litteraire     <class 'str'>    autre
texte                <class 'str'>    Ans sur cette quatrième Edition'. -S quelque temps, il a paru de nouvelles critiques d'Atala. Je n'a
tokens               <class 'list'>    ['Ans', 'sur', 'cette', 'quatrième', 'Edition', "'", '.', '-S', 'quelque', 'temps', ',', 'il', 'a', 
lemmes               <class 'list'>    ['an', 'sur', 'ce', 'quatrième', 'edition', "'", '.', '-s', 'quelque', 'temps', ',', 'il', 'avoir', 


### Vocabulaire

Construire un dictionnaire du vocabulaire du corpus.

### Choix de l'unité élémentaire : tokens ou lemmes ?

In [7]:
for word_type in ["tokens", "lemmes"]:
    all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
    vocab = sorted(set(all_words))
    print(f"{word_type:<10}  Taille du vocabulaire: {len(vocab):,} words")

tokens      Taille du vocabulaire: 322,838 words
lemmes      Taille du vocabulaire: 285,868 words


Nous retenons les lemmes pour réduire la taille du vocabulaire

In [8]:
word_type = "lemmes" # ["lemmes", "tokens"]

In [9]:
all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
vocab = sorted(set(all_words))


print(f"Documents : {len(dataset):,}")
print(f"Taille du corpus : {len(all_words):,}")
print(f"Mots / Document : {int(len(all_words)/len(dataset)):,}")
print(f"Taille du vocabulaire : {len(vocab):,} ({word_type})")

Documents : 200
Taille du corpus : 19,719,818
Mots / Document : 98,599
Taille du vocabulaire : 285,868 (lemmes)


In [10]:
import pandas as pd
from collections import Counter

vocab_counter = Counter(all_words).most_common()
# vocab = list(dict(vocab_counter).keys())

df = pd.DataFrame([
    {
        "Rang": i + 1,
        "Word": word,
        "Occurrences": count,
        "Fréquence (%)": 100 * count / len(all_words)
    }
    for i, (word, count) in enumerate(vocab_counter)
])

df

,Rang,Word,Occurrences,Fréquence (%)
0,1,",",1337995,6.785027
1,2,le,1252190,6.349906
2,3,de,1067224,5.411936
3,4,.,693278,3.515641
4,5,un,434497,2.203352
...,...,...,...,...
285863,285864,mous,1,0.000005
285864,285865,silenciesuement,1,0.000005
285865,285866,téqui,1,0.000005
285866,285867,1948,1,0.000005


In [11]:
word_lists = [
    [word.lower() for word in doc[word_type]]
    for doc in dataset
]
word_counts = Counter(all_words)

print(f"Vocabulaire : {len(vocab_counter):,}")
print(f"Mots apparaissant une seule fois : {sum(c[1] == 1 for c in vocab_counter):,}")
print(f"Mots apparaissant 5 fois ou plus : {sum(c[1] >= 5 for c in vocab_counter):,}")
print(f"Mots apparaissant 20 fois ou plus : {sum(c[1] >= 20 for c in vocab_counter):,}")
print(f"Mots apparaissant 100 fois ou plus : {sum(c[1] >= 100 for c in vocab_counter):,}")
print(f"Mots apparaissant 1 000 fois ou plus : {sum(c[1] >= 1000 for c in vocab_counter):,}")

Vocabulaire : 285,868
Mots apparaissant une seule fois : 189,014
Mots apparaissant 5 fois ou plus : 43,203
Mots apparaissant 20 fois ou plus : 20,367
Mots apparaissant 100 fois ou plus : 7,926
Mots apparaissant 1 000 fois ou plus : 1,528


In [12]:
import numpy as np
from scipy.sparse import coo_matrix
from tqdm import tqdm

# Context Window
N = 3

word_to_id = {word: i for i, word in enumerate(vocab)}

rows = []
cols = []

for tokens in tqdm(word_lists):

    ids = np.fromiter(
        (word_to_id[w] for w in tokens if w in word_to_id),
        dtype=np.int32
    )

    # Generate context pairs for each distance
    for d in range(1, N + 1):

        if len(ids) <= d:
            continue

        left = ids[:-d]
        right = ids[d:]

        # Both directions: target -> context and context -> target
        rows.append(np.concatenate((left, right)))
        cols.append(np.concatenate((right, left)))

# Concatenate all pairs
rows = np.concatenate(rows)
cols = np.concatenate(cols)

# Build sparse matrix
cooc_matrix = coo_matrix(
    (
        np.ones(len(rows), dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(vocab), len(vocab))
).tocsr()

print("Dimensions de la matrice :", cooc_matrix.shape)
print(f"Nombre d'entrées non nulles : {cooc_matrix.nnz:,} ({cooc_matrix.nnz/(cooc_matrix.shape[0]*cooc_matrix.shape[1]):.4%})")

100%|██████████| 200/200 [00:05<00:00, 36.67it/s]


Dimensions de la matrice : (285868, 285868)
Nombre d'entrées non nulles : 12,754,864 (0.0156%)


In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_most_similar(
    word,
    matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=1
):

    if word not in word_to_id:
        print(f"'{word}' is not in the vocabulary.")
        return

    word_id = word_to_id[word]

    # Compute similarity against all word vectors
    similarities = cosine_similarity(
        matrix[word_id],
        matrix
    ).flatten().tolist()

    # Rank candidates
    closest_ids = np.argsort(similarities)[::-1]

    results = []

    for idx in closest_ids:

        candidate = vocab[idx]

        # Exclude the query word
        if idx == word_id:
            continue

        # Apply frequency threshold
        if word_counts[candidate] < min_count:
            continue

        results.append((candidate, round(similarities[idx], 4)))

        if len(results) == top_n:
            break

    return results

In [14]:
min_word_occurrences = 0

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]
print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

Vocabulaire après filtrage: 285,868


In [15]:
target_word = "france"
most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=min_word_occurrences
)
print(f"Mot Cible  : {target_word}")
print(f"Contexte N : {N}")
print(f"------------------")
print(pd.DataFrame(most_similar, columns=["Word", "Cos_Sim"]).to_string(index=False, header=False, justify=["right", "center"]))

Mot Cible  : france
Contexte N : 3
------------------
  bretagne 0.9902
    europe 0.9899
angleterre 0.9891
    italie 0.9859
    orient 0.9845
   espagne 0.9844
    russie 0.9840
    egypte 0.9839
    sicile 0.9815
    savoie 0.9811


In [16]:
for target_word in ["lundi", "fusil", "joue", "maison", "roi", "tristesse", "sept", "france", "robe", "car"]:
   most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=5,
    min_count=0)
   print(f"{target_word:<10} -> {most_similar[0][0]:<10} ->  {most_similar[1][0]:<10} ->  {most_similar[2][0]:<10} ->  {most_similar[3][0]:<10} ->  {most_similar[4][0]:<10}")

lundi      -> dimanche   ->  mardi      ->  vendredi   ->  jeudi      ->  samedi    
fusil      -> sabre      ->  bâton      ->  couteau    ->  carabine   ->  pistolet  
joue       -> front      ->  lèvre      ->  visage     ->  poitrine   ->  jambe     
maison     -> ville      ->  société    ->  grotte     ->  intérieur  ->  parc      
roi        -> gouverneur ->  chevalier  ->  ministre   ->  cardinal   ->  peuple    
tristesse  -> angoisse   ->  mélancolie ->  joie       ->  terreur    ->  embarras  
sept       -> neuf       ->  quatre     ->  dix        ->  cinq       ->  six       
france     -> bretagne   ->  europe     ->  angleterre ->  italie     ->  orient    
robe       -> tunique    ->  bonnet     ->  costume    ->  habit      ->  armure    
car        -> puisque    ->  quoique    ->  sinon      ->  peut-être  ->  non-seulement


## Réduction de dimension et Visualisation 2D

In [20]:
min_word_occurrences = 500 # filtre le vocabulaire à garder pour la visualisation

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

Vocabulaire après filtrage: 2,606


In [21]:
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]
filtered_word_to_id = {word: i for i, word in enumerate(filtered_vocab)}

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)

Original matrix: (285868, 285868)
Filtered matrix: (2606, 285868)


### Réduction de dimensions avec UMAP

In [22]:
import umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=25,
    min_dist=0.1,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)

# 2. Mean-center the entire 3D projection around (0, 0, 0)
embedding = embedding - embedding.mean(axis=0)

/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)
/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


In [23]:
import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    f"{word_type}": filtered_vocab
})

In [24]:
target_words = ["père", "mère", "rêver"]

embedding_df["group"] = embedding_df[word_type].isin(target_words).map({
    True: "Target",
    False: "Other words"
})

In [25]:
fig = px.scatter(
    embedding_df,
    x="x",
    y="y",
    color="group",
    hover_name=f"{word_type}",
    title="UMAP Visualization of French Word Co-occurrences",
    opacity=0.8,
)

fig.update_traces(
    marker=dict(size=5)
)

fig.update_layout(
    width=1000,
    height=800,
    template="plotly_white",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
)

fig.show()

### Similarité en 3 dimensions

In [27]:
min_word_occurrences = 500

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)


import umap

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)
embedding = embedding - embedding.mean(axis=0)


import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    f"{word_type}": filtered_vocab
})

Vocabulary after filtering: 2,606
Original matrix: (285868, 285868)
Filtered matrix: (2606, 2606)


/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


In [28]:
import numpy as np
import pandas as pd
import umap
from sklearn.preprocessing import normalize

# 1. Filter vocabulary and co-occurrence matrix
min_word_occurrences = 500
filtered_vocab = [w for w, c in word_counts.items() if c >= min_word_occurrences]
filtered_ids = [word_to_id[w] for w in filtered_vocab]
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids].astype("float32")

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
print(f"Matrix shape: {cooc_matrix.shape} -> {filtered_matrix.shape}\n")

# 2. UMAP dimensionality reduction + Mean-centering
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
)
embedding = reducer.fit_transform(filtered_matrix)
embedding -= embedding.mean(axis=0)

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    word_type: filtered_vocab
})

# 3. Extract target words, normalize, and compute cosine similarity
target_words = ["père", "mère", "rêver"]
target_df = embedding_df[embedding_df[word_type].isin(target_words)]

words = target_df[word_type].values
vectors = target_df[["x", "y", "z"]].values
norm_vectors = normalize(vectors, norm="l2")

print("--- Centered & Normalized Vectors (3D) ---")
for word, vec in zip(words, norm_vectors):
    print(f"{word:<8}: {np.round(vec, 4)}")

sim_df = pd.DataFrame(norm_vectors @ norm_vectors.T, index=words, columns=words)

print("\n--- Cosine Similarity Matrix ---")
print(sim_df.round(4))

Vocabulary after filtering: 2,606
Matrix shape: (285868, 285868) -> (2606, 2606)



/home/antoine/cours/DAUPHINE_2026-2027/.venv/lib/python3.12/site-packages/umap/umap_.py:4160: FutureWarning: compatibility_layout was not set, so UMAP is using the legacy (0.5.x) nearest-neighbor, initialization and layout pipeline. In a future release the default will change to compatibility_layout=False, which uses recursive initialization and the Adam optimizer and produces different embeddings. Set compatibility_layout=True to keep the current behavior and silence this warning, or compatibility_layout=False (or optimizer='adam') to adopt the new behavior now.
  self.fit(X, y, ensure_all_finite, **kwargs)


--- Centered & Normalized Vectors (3D) ---
père    : [ 0.2772 -0.8605  0.4273]
mère    : [ 0.301  -0.8043  0.5124]
rêver   : [-0.9051 -0.4242 -0.028 ]

--- Cosine Similarity Matrix ---
         père    mère   rêver
père   1.0000  0.9945  0.1022
mère   0.9945  1.0000  0.0544
rêver  0.1022  0.0544  1.0000


In [29]:
sim_df

,père,mère,rêver
père,1.000000,0.994520,0.102153
mère,0.994520,1.000000,0.054402
rêver,0.102153,0.054402,1.000000


## Évaluer un modèle de similarité

In [55]:
import pandas as pd

similarity_benchmarks = {
    "MC": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-mc.dataset",
    "RG": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-rg.dataset",
    "SimLex": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-simlex.dataset",
    "WS353": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-ws353.dataset",
}

datasets = {}

for name, url in similarity_benchmarks.items():
    df = pd.read_csv(url, sep=";")

    df["dataset"] = name
    df = df[["dataset", "word1", "word2", "score"]].copy()

    df["word1"] = df["word1"].str.lower()
    df["word2"] = df["word2"].str.lower()

    # Ensure word1 (A) comes before word2 (B) alphabetically
    df[["word1", "word2"]] = df[["word1", "word2"]].apply(
        lambda row: sorted(row), axis=1, result_type="expand"
    )

    datasets[name] = df

    print(f"{name}: {df.shape}")

# Merge all datasets into a single DataFrame
df_all = pd.concat(datasets.values(), ignore_index=True)

# Remove duplicate word pairs (regardless of dataset)
# df_all = df_all.drop_duplicates(subset=["word1", "word2"], keep="first")

# Reset index
df_all = df_all.reset_index(drop=True)

print(f"All datasets merged: {df_all.shape}")

MC: (30, 4)
RG: (65, 4)
SimLex: (999, 4)
WS353: (353, 4)
All datasets merged: (1447, 4)


In [56]:
df_all

,dataset,word1,word2,score
0,MC,asile,refuge,3.61
1,MC,auto,automobile,3.92
2,MC,grue,oiseau,3.05
3,MC,coq,oiseau,2.97
4,MC,gars,garçon,3.76
...,...,...,...,...
1442,WS353,prévision,temps,8.34
1443,WS353,mercredi,nouvelles,2.22
1444,WS353,bois,forêt,7.73
1445,WS353,mot,similarité,4.75


In [57]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def predict_similarity(word1, word2, matrix, word_to_id):

    if word1 not in word_to_id or word2 not in word_to_id:
        return np.nan

    id1 = word_to_id[word1]
    id2 = word_to_id[word2]

    similarity = cosine_similarity(
        matrix[id1],
        matrix[id2]
    )[0, 0]

    return float(similarity)

def evaluate_similarity(dataset, matrix, word_to_id):

    human_scores = []
    model_scores = []

    for _, row in dataset.iterrows():

        word1 = row["word1"]
        word2 = row["word2"]

        if word1 not in word_to_id or word2 not in word_to_id:
            continue

        id1 = word_to_id[word1]
        id2 = word_to_id[word2]

        similarity = cosine_similarity(
            matrix[id1],
            matrix[id2]
        )[0, 0]

        human_scores.append(row["score"])
        model_scores.append(float(similarity))

    correlation, p_value = spearmanr(
        human_scores,
        model_scores
    )

    coverage = len(human_scores) / len(dataset)

    return correlation, p_value, coverage

In [58]:
results = {}

for name, dataset in datasets.items():

    correlation, p_value, coverage = evaluate_similarity(
        dataset,
        cooc_matrix,
        word_to_id
    )

    results[name] = {
        "Spearman": correlation,
        "p-value": p_value,
        "Coverage": coverage
    }

results_df = pd.DataFrame(results).T

print(results_df.round(4))

        Spearman  p-value  Coverage
MC        0.4831   0.0068    1.0000
RG        0.5356   0.0000    0.9692
SimLex    0.0812   0.0170    0.8629
WS353     0.1068   0.0638    0.8555


In [59]:
df_all["prediction"] = df_all.apply(
    lambda row: predict_similarity(
        row["word1"],
        row["word2"],
        cooc_matrix,
        word_to_id
    ),
    axis=1
)

In [60]:
from sklearn.preprocessing import MinMaxScaler

df_all["score_scaled"] = (
    df_all.groupby("dataset")["score"]
    .transform(lambda x: MinMaxScaler().fit_transform(x.values.reshape(-1, 1)).ravel())
)

df_all["prediction_scaled"] = (
    df_all.groupby("dataset")["prediction"]
    .transform(lambda x: MinMaxScaler().fit_transform(x.values.reshape(-1, 1)).ravel())
)

In [61]:
df_all["error"] = (
    df_all["score_scaled"] - df_all["prediction_scaled"]
)

df_all["absolute_error"] = df_all["error"].abs()

df_all["squared_error"] = df_all["error"] ** 2

In [62]:
error_results = df_all.groupby("dataset").agg(
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean"),
    RMSE=("squared_error", lambda x: x.mean() ** 0.5),
    N=("absolute_error", "count")
)

print(error_results.round(4))

            MAE     MSE    RMSE    N
dataset                             
MC       0.3191  0.1829  0.4276   30
RG       0.4503  0.2527  0.5027   63
SimLex   0.5305  0.3503  0.5919  862
WS353    0.3025  0.1381  0.3716  302


In [69]:
df_all.sort_values(by="absolute_error", ascending=False)

,dataset,word1,word2,score,prediction,score_scaled,prediction_scaled,error,absolute_error,squared_error
228,SimLex,chat,chien,0.00,0.983722,0.000000,0.981325,-0.981325,0.981325,0.962999
107,SimLex,difficile,facile,0.00,0.982554,0.000000,0.979984,-0.979984,0.979984,0.960370
220,SimLex,femme,homme,0.00,0.978718,0.000000,0.975584,-0.975584,0.975584,0.951764
209,SimLex,journée,nuit,0.00,0.977802,0.000000,0.974533,-0.974533,0.974533,0.949714
210,SimLex,nord,sud,0.08,0.981692,0.008000,0.978996,-0.970996,0.970996,0.942833
...,...,...,...,...,...,...,...,...,...,...
1431,WS353,récupération,traitement,7.91,NaN,0.786080,NaN,NaN,NaN,NaN
1434,WS353,archives,vidéo,6.34,NaN,0.625384,NaN,NaN,NaN,NaN
1435,WS353,en série,téléspectateur,2.97,NaN,0.280450,NaN,NaN,NaN,NaN
1445,WS353,mot,similarité,4.75,NaN,0.462641,NaN,NaN,NaN,NaN


In [75]:
datasets["SimLex"]

,dataset,word1,word2,score
0,SimLex,nouveau,vieux,0.00
1,SimLex,intelligent,intelligent,10.00
2,SimLex,difficile,dur,9.69
3,SimLex,content,de bonne humeur,9.31
4,SimLex,dur,facile,0.00
...,...,...,...,...
994,SimLex,acquérir,rejoindre,2.15
995,SimLex,assister,envoyer,2.08
996,SimLex,assister,rassembler,2.85
997,SimLex,absorber,se désister,1.08


### Simlex encode strictement la synonymie, et non simpement l'appartenance au même champ lexical.

### Limite des approches distributionnelles (cooccurrence)

In [76]:
datasets["WS353"]

,dataset,word1,word2,score
0,WS353,admission,billet,7.69
1,WS353,alcool,chimie,5.54
2,WS353,aluminium,métal,7.83
3,WS353,annonce,effort,2.75
4,WS353,annonce,nouvelles,7.56
...,...,...,...,...
348,WS353,prévision,temps,8.34
349,WS353,mercredi,nouvelles,2.22
350,WS353,bois,forêt,7.73
351,WS353,mot,similarité,4.75
